# Pathway And Gene Set Analysis

**REQUIRED DAY 3**

## From a gene list to a biological story

Lesson 04 gave you a table of ~10,000 tested genes, thousands of them significant. That's not yet an answer to "what is IFN-β actually doing to these monocytes" — it's a list. Pathway enrichment asks which *coherent biological processes* those genes cluster into, rather than making you read 3,411 gene names one at a time.

## The gene sets: already fetched, already cached

`decoupler.op.hallmark()` pulls the MSigDB Hallmark collection (50 well-curated pathways) from a remote resource at call time — the same live-network-call problem Day 2's FASTQs and today's Kang dataset both had. It's already been fetched once and cached at `/tscc/nfs/home/juf009/day3_shared_data/hallmark_genesets.csv`.

In [ ]:
import pandas as pd
import decoupler as dc

hallmark = pd.read_csv("/tscc/nfs/home/juf009/day3_shared_data/hallmark_genesets.csv")
print(hallmark.shape, "gene-pathway pairs across", hallmark["source"].nunique(), "pathways")
hallmark.head()

## Enrichment on the real DE results from lesson 04

`decoupler` expects a table shaped like samples (rows) x genes (columns), with a per-gene statistic. We have one "sample" -- the CD14+ Monocytes stim-vs-ctrl contrast -- so it's a 1-row table using `log2FoldChange` as the statistic. Loaded from the real file lesson 04 saved (`results/de_results_cd14mono.csv`), not from in-memory state, so this notebook runs on its own.

In [ ]:
import pandas as pd

pb_results = pd.read_csv("results/de_results_cd14mono.csv", index_col=0)

stat_df = pb_results[["log2FoldChange"]].dropna().T
stat_df.index = ["CD14_Monocytes_stim_vs_ctrl"]
stat_df.shape  # (1 sample, ~10000 genes)


In [ ]:
es, pv = dc.mt.ora(stat_df, hallmark, tmin=5)
result = pd.DataFrame({"pathway": es.columns, "score": es.iloc[0].values, "pval": pv.iloc[0].values})
result = result.sort_values("pval")
result.head(15)

## The real result, and why it matters

The top two hits on this exact data are **INTERFERON_GAMMA_RESPONSE** (score 6.01) and **INTERFERON_ALPHA_RESPONSE** (score 5.31) — out of 50 possible pathways. That's not a coincidence: IFN-β is a type I interferon, closely related to IFN-α, and type I interferon signaling strongly cross-activates IFN-γ response genes. This is the same kind of sanity check as lesson 04's ISG15/IFIT1 check — when the biology is this strong, a correct pipeline should recover it, and it did, all the way from raw counts through pseudobulk DE through enrichment.

## The pitfall: trying gene sets until one works

`decoupler` supports many gene-set collections and several enrichment methods (`dc.mt.ora`, `dc.mt.gsea`, `dc.mt.ulm`, and others) beyond what's shown here. Nothing stops you from running all of them and reporting whichever produced the most flattering result — which is exactly Agent-B checklist item 24. Decide your gene-set database and method *before* looking at the output, and report that decision alongside the result, the same discipline as pre-registering a hypothesis.

## Agent-assisted pathway analysis, done right

> Weak: "Find enriched pathways in this DE result."
>
> Strong: "Run over-representation analysis on this pseudobulk DE result using the Hallmark gene sets and ORA specifically — state that choice before showing me results, don't try multiple databases or methods and report whichever looks most interesting."

## Practice

Run the enrichment yourself and confirm the two interferon pathways top the list. Open a fresh Agent B session and run checklist items 20 (correction matches the actual test count — 50 pathways here, not one) and 24 (gene-set/method fixed in advance) against this notebook.

## Further reading

- [decoupler documentation](https://decoupler.readthedocs.io/)
- [Single-cell best practices — Gene set enrichment and pathway analysis](https://www.sc-best-practices.org/conditions/gsea_pathway.html)
- [MSigDB Hallmark collection](https://www.gsea-msigdb.org/gsea/msigdb/collections.jsp#H)